In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os, time, random, json
import numpy as np
import urllib.request, zipfile

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    "num_classes":      46,
    "image_size":       32,
    "batch_size":       64,
    "epochs":           100,
    "lr":               5e-4,
    "weight_decay":     1e-4,
    "label_smoothing":  0.1,
    "val_split":        0.1,
    "data_dir":         "/kaggle/input/datasets/rautranjit/devanagari/DevanagariHandwrittenCharacterDataset",
    "results_dir":      "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

NUM_CLASSES = CFG["num_classes"]
IMG         = CFG["image_size"]
BS          = CFG["batch_size"]
AUTOTUNE    = tf.data.AUTOTUNE

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
zip_path = "/kaggle/input/datasets/theranjitraut/devanagari/DevanagariHandwrittenCharacterDataset"

if os.path.exists(CFG["data_dir"]):
    print("[INFO] DHCD already present – skipping download.")
else:
    print("[INFO] Downloading DHCD …")
    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall("./data/")
        os.rename("./data/DevanagariHandwrittenCharacterDataset", CFG["data_dir"])
        print("[INFO] DHCD extracted successfully.")
    except Exception as exc:
        print(f"[WARN] Download failed: {exc}")

train_full = keras.utils.image_dataset_from_directory(
    os.path.join(CFG["data_dir"], "Train"),
    image_size=(IMG, IMG), batch_size=None,
    color_mode="grayscale", label_mode="int", seed=SEED,
)
test_ds_raw = keras.utils.image_dataset_from_directory(
    os.path.join(CFG["data_dir"], "Test"),
    image_size=(IMG, IMG), batch_size=None,
    color_mode="grayscale", label_mode="int", seed=SEED,
)

total   = tf.data.experimental.cardinality(train_full).numpy()
n_val   = max(1, int(total * CFG["val_split"]))
n_train = total - n_val
train_ds_raw = train_full.take(n_train)
val_ds_raw   = train_full.skip(n_train)

def normalise(img, lbl):
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    return img, lbl

def augment(img, lbl):
    img = tf.image.random_brightness(img, 0.2)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.pad(img, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    img = tf.image.random_crop(img, [IMG, IMG, 1])
    return img, lbl

def to_onehot(img, lbl):
    return img, tf.one_hot(lbl, NUM_CLASSES)

train_ds = (
    train_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .shuffle(8192, seed=SEED)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
val_ds = (
    val_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds = (
    test_ds_raw
    .map(normalise, num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)
test_ds_oh = (
    test_ds_raw
    .map(normalise,  num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS)
    .prefetch(AUTOTUNE)
)

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: (batched)")

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset": "\033[0m", "bold": "\033[1m", "cyan": "\033[96m",
    "yellow": "\033[93m", "green": "\033[92m", "red": "\033[91m",
    "grey": "\033[90m", "white": "\033[97m", "blue": "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def print_model_summary(model: Model) -> None:
    W = 62
    trainable     = model.count_params()
    non_trainable = sum(tf.size(w).numpy() for w in model.non_trainable_weights)
    total = trainable + non_trainable
    title = f"  {model.name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    hdr = f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║"
    print(_c(hdr, "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    shown_layers = [l for l in model.layers if l.count_params() > 0][:20]
    for lyr in shown_layers:
        n_params = lyr.count_params()
        row = f"║  {lyr.name[:14]:<16}║  {type(lyr).__name__[:21]:<21}║  {n_params:>15,}  ║"
        print(row)
    if len([l for l in model.layers if l.count_params() > 0]) > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

class EpochProgressCallback(keras.callbacks.Callback):
    BAR_WIDTH = 20
    def __init__(self, total_epochs: int, model_name: str):
        super().__init__()
        self.total_epochs = total_epochs
        self.model_name   = model_name
        self._epoch_start = 0.0
    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = time.time()
    def on_epoch_end(self, epoch, logs=None):
        logs    = logs or {}
        elapsed = time.time() - self._epoch_start
        ep_num  = epoch + 1
        pct     = ep_num / self.total_epochs
        filled  = int(self.BAR_WIDTH * pct)
        bar     = "█" * filled + "░" * (self.BAR_WIDTH - filled)
        loss    = logs.get("loss",         float("nan"))
        acc     = logs.get("accuracy",     float("nan")) * 100
        val_acc = logs.get("val_accuracy", float("nan")) * 100
        val_los = logs.get("val_loss",     float("nan"))
        try:
            lr_val = float(keras.backend.get_value(self.model.optimizer.learning_rate))
            lr_str = f"lr={lr_val:.2e}"
        except Exception:
            lr_str = ""
        print(
            f"  {_c(f'Epoch {ep_num:>3}/{self.total_epochs}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={loss:.4f}', 'white')}  {_c(f'acc={acc:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_los:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc:.2f}%', 'yellow' if val_acc < acc else 'green')}  "
            f"{_c(lr_str, 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

def print_comparison_table(results: dict) -> None:
    W = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    hdr = f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║"
    print(_c(hdr, "bold", "white"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (
            f"║{star} {name:<22}║{r['params']:>10,} ║"
            f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║"
            f"{r['test_loss']:>5.3f} ║"
        )
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n", "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    """GELU activation – smoother than ReLU, better gradients in deep nets."""
    return tf.nn.gelu(x)

def residual_block(x, channels: int):
    """Standard pre-activation residual block: Conv→BN→GELU→Conv→BN→Add→GELU."""
    residual = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, residual])
    x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_channels: int, out_channels: int):
    """
    DenseNet-inspired block: 2 residual blocks with dense concat,
    bottleneck 1×1 projection, then stride-2 depthwise downsampling.
    """
    if in_channels != out_channels:
        skip = layers.Conv2D(out_channels, 1, use_bias=False)(x)
        skip = layers.BatchNormalization()(skip)
        x_in = layers.Activation(gelu)(skip)
    else:
        x_in = x
    r1  = residual_block(x_in, out_channels)
    r2  = residual_block(r1,   out_channels)
    cat = layers.Concatenate()([r1, r2])
    out = layers.Conv2D(out_channels, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_channels, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out)
    out = layers.Activation(gelu)(out)
    return out

def channel_attention(x, channels: int, reduction: int = 8):
    """Squeeze-and-Excitation channel attention."""
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(channels // reduction, activation="gelu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def adaptive_filter_capsule(x, num_classes: int, capsule_dim: int = 16):
    """
    Lightweight capsule-like routing module.
    Projects features into (num_classes × capsule_dim) space,
    filters per-class, and sums to produce class-discriminative scores.
    No dynamic routing – O(n) cost.
    """
    h = layers.Dense(256, activation=gelu)(x)
    h = layers.Dense(num_classes * capsule_dim)(h)
    h = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps = layers.Multiply()([x_sliced, h])
    caps = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    caps = layers.BatchNormalization()(caps)
    return caps

def stroke_topology_module(x, out_features: int):
    """
    Devanagari-specific stroke topology encoder.

    Uses asymmetric kernels (1×5, 5×1) to capture horizontal and vertical
    strokes (including the Devanagari shirorekha top-bar), complemented by
    two parallel dilated convolutions (rate 1 and 2) for local and semi-local
    context. The four feature maps are concatenated, pooled, and projected to
    a fixed-size vector of dimension `out_features`.

    Input:  spatial feature map  (B, H, W, C)  — typically enc3 output
    Output: stroke feature vector (B, out_features)
    """
    h  = layers.Conv2D(64, (1, 5), padding="same", use_bias=False, activation=gelu)(x)
    v  = layers.Conv2D(64, (5, 1), padding="same", use_bias=False, activation=gelu)(x)
    d1 = layers.Conv2D(32, 3, padding="same", dilation_rate=1, use_bias=False, activation=gelu)(x)
    d2 = layers.Conv2D(32, 3, padding="same", dilation_rate=2, use_bias=False, activation=gelu)(x)
    out = layers.Concatenate()([h, v, d1, d2])
    out = layers.BatchNormalization()(out)
    out = layers.GlobalAveragePooling2D()(out)
    out = layers.Dense(out_features, activation=gelu)(out)
    return out

# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITIONS
# ─────────────────────────────────────────────────────────────────────────────
def build_our_model_net(
    num_classes: int = 46,
    image_size:  int = 32,
    dropout_rate: float = 0.3,
    weight_decay: float = 1e-4,
    head_units:   int   = 256,
) -> Model:
    """
    our_model-Net + STM Experimental Variant.

    Architecture overview
    ─────────────────────
    Stem (dual-path):
      • Standard 3×3 conv path
      • Horizontal stroke scaffold (1×5 conv)
      → Concatenated + SE channel attention + 1×1 projection

    Encoder (3 stages, each halving spatial dims):
      enc1: 64→64    (16×16 after stride-2 DW)
      enc2: 64→128   ( 8× 8)
      enc3: 128→256  ( 4× 4)
      Each stage injects a pooled scaffold residual (weight=0.1).

    Decoder head — THREE parallel streams:
      ① Dense head  : fused_gap → Dense(256) → LayerNorm → logits_dense  (B, K)
      ② AFC stream  : fused_gap → AdaptiveFilterCapsule               → afc_out   (B, K)
      ③ STM stream  : enc3 (spatial) → StrokeTopologyModule(256)
                       → Dense(K)                                      → stm_out   (B, K)

    Gated fusion (3-way softmax):
      gate = Dense(3, softmax)(concat[logits_dense, afc_out, stm_out])
      output = gate[:,0]*logits_dense + gate[:,1]*afc_out + gate[:,2]*stm_out

    Changes vs. Paper 1 baseline
    ─────────────────────────────
    • Added STM branch operating on enc3 spatial maps (not the pooled vector).
    • Gate expanded from 2-way to 3-way to accommodate the new STM stream.
    • Everything else (stem, encoder, AFC, dense head) is identical to Paper 1.
    """
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────────
    # Texture path
    t       = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t       = layers.BatchNormalization()(t)
    t       = layers.Activation(gelu)(t)
    # Stroke scaffold (horizontal asymmetric conv for shirorekha detection)
    s       = layers.Conv2D(32, (1, 5), padding="same", use_bias=False)(inp)
    s       = layers.BatchNormalization()(s)
    scaffold = layers.Activation(gelu)(s)

    stem = layers.Concatenate()([t, scaffold])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem)
    stem = layers.Activation(gelu)(stem)

    # ── Encoder ───────────────────────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64, 64)
    sc1  = layers.AveragePooling2D(2)(layers.Conv2D(64,  1, use_bias=False)(scaffold))
    enc1 = layers.Add()([enc1, layers.Lambda(lambda t: t * 0.1)(sc1)])

    enc2 = dense_res_block(enc1, 64, 128)
    sc2  = layers.AveragePooling2D(4)(layers.Conv2D(128, 1, use_bias=False)(scaffold))
    enc2 = layers.Add()([enc2, layers.Lambda(lambda t: t * 0.1)(sc2)])

    enc3 = dense_res_block(enc2, 128, 256)
    sc3  = layers.AveragePooling2D(8)(layers.Conv2D(256, 1, use_bias=False)(scaffold))
    enc3 = layers.Add()([enc3, layers.Lambda(lambda t: t * 0.1)(sc3)])

    # ── Multi-scale GAP fusion (unchanged from Paper 1) ───────────────────────
    gap1 = layers.GlobalAveragePooling2D(name="gap1")(enc1)
    gap2 = layers.GlobalAveragePooling2D(name="gap2")(enc2)
    gap3 = layers.GlobalAveragePooling2D(name="gap3")(enc3)
    fused_gap = layers.Concatenate(name="multiscale_fused")([gap1, gap2, gap3])

    # ── Stream ①: Dense head (unchanged from Paper 1) ────────────────────────
    x = layers.Dense(head_units, use_bias=False, name="head_dense")(fused_gap)
    x = layers.LayerNormalization(name="head_ln")(x)
    x = layers.Activation("gelu", name="head_act")(x)
    logits_dense = layers.Dense(K, name="head_logits")(x)         # (B, K)

    # ── Stream ②: Adaptive Filter Capsule (unchanged from Paper 1) ───────────
    afc_out = adaptive_filter_capsule(fused_gap, K)                # (B, K)

    # ── Stream ③: Stroke Topology Module (NEW) ────────────────────────────────
    # Operates on enc3 *spatial maps* (not pooled), so it retains spatial
    # stroke structure that GAP would otherwise discard.
    # The STM output is a 256-d stroke feature vector, then projected to K logits.
    stm_vec = stroke_topology_module(enc3, out_features=256)       # (B, 256)
    stm_out = layers.Dense(K, name="stm_logits")(stm_vec)         # (B, K)

    # ── 3-way gated fusion ────────────────────────────────────────────────────
    # Gate input: concatenation of all three logit streams.
    # A 3-way softmax produces per-sample mixing weights:
    #   gate[:,0] → weight for dense head
    #   gate[:,1] → weight for AFC
    #   gate[:,2] → weight for STM
    #
    # This is the minimal change over Paper 1 (which used a 2-way gate over
    # dense + AFC).  Adding STM here lets the model learn dynamically whether
    # stroke topology information improves the prediction for each sample.
    gate_input = layers.Concatenate(name="gate_input")(
        [logits_dense, afc_out, stm_out]
    )
    gate = layers.Dense(3, activation="softmax", name="gate")(gate_input)  # (B, 3)

    dense_scaled = layers.Lambda(
        lambda t: t[0] * t[1][:, 0:1], name="gate_dense"
    )([logits_dense, gate])

    afc_scaled = layers.Lambda(
        lambda t: t[0] * t[1][:, 1:2], name="gate_afc"
    )([afc_out, gate])

    stm_scaled = layers.Lambda(
        lambda t: t[0] * t[1][:, 2:3], name="gate_stm"
    )([stm_out, gate])

    outputs = layers.Add(name="logits")([dense_scaled, afc_scaled, stm_scaled])

    model = keras.Model(inputs=inp, outputs=outputs, name="our_model-Net-STM")
    return model

# Registry
MODELS_TF = {
    "our_model-Net-STM": lambda: build_our_model_net(NUM_CLASSES, IMG),
}

# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealing(keras.optimizers.schedules.LearningRateSchedule):
    """Cosine-annealing without restarts: lr(t) = max(base·0.5·(1+cos(π·t/T)), 1e-6)"""
    def __init__(self, base: float, steps: int):
        self.base  = base
        self.steps = tf.cast(steps, tf.float32)
    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * step / self.steps))
        return tf.maximum(self.base * cosine, 1e-6)
    def get_config(self):
        return {"base": self.base, "steps": int(self.steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def compile_model(model: Model, steps_total: int) -> Model:
    lr_sch = CosineAnnealing(CFG["lr"], steps_total)
    model.compile(
        optimizer=keras.optimizers.AdamW(
            learning_rate=lr_sch,
            weight_decay=CFG["weight_decay"],
        ),
        loss=keras.losses.CategoricalCrossentropy(
            from_logits=True,
            label_smoothing=CFG["label_smoothing"],
        ),
        metrics=["accuracy"],
        jit_compile=True,
    )
    return model

def compute_macro_f1(model: Model, dataset) -> float:
    tp = np.zeros(NUM_CLASSES)
    fp = np.zeros(NUM_CLASSES)
    fn = np.zeros(NUM_CLASSES)
    for images, labels in dataset:
        preds = tf.argmax(model(images, training=False), axis=1).numpy()
        lbls  = labels.numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. TRAIN + EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
trained_models  = {}
all_histories   = {}
steps_per_epoch = sum(1 for _ in train_ds)
total_steps     = steps_per_epoch * CFG["epochs"]

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS_TF)} models × {CFG['epochs']} epochs", "bold", "white"))
print(_c(f"{'═'*70}\n", "cyan"))

for name, model_fn in MODELS_TF.items():
    model = model_fn()
    model = compile_model(model, total_steps)
    print_model_summary(model)

    ckpt_path = os.path.join(CFG["results_dir"], f"{name}_best.keras")
    callbacks = [
        ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=15, restore_best_weights=True, verbose=0),
        # EpochProgressCallback(CFG["epochs"], name),
    ]

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    t0 = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=CFG["epochs"],
        callbacks=callbacks,
        verbose=1,
    )
    elapsed  = time.time() - t0
    best_val = max(history.history["val_accuracy"]) * 100.0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )
    trained_models[name] = model
    all_histories[name]  = history.history

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
results = {}
for name, model in trained_models.items():
    test_loss, test_acc_raw = model.evaluate(test_ds_oh, verbose=0)
    test_acc = test_acc_raw * 100.0
    macro_f1 = compute_macro_f1(model, test_ds)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    model.count_params(),
        "test_loss": round(float(test_loss), 4),
    }

print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "tensorflow_results_stm.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results  → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "tensorflow_histories_stm.json")
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] TensorFlow STM benchmark complete.\n", "green", "bold"))

[INFO] DHCD already present – skipping download.
Found 78200 files belonging to 46 classes.
Found 13800 files belonging to 46 classes.
[INFO] Train: 70,380 | Val: 7,820 | Test: (batched)

══════════════════════════════════════════════════════════════════════
  Starting benchmark: 1 models × 100 epochs
══════════════════════════════════════════════════════════════════════


╔══════════════════════════════════════════════════════════════╗
║  our_model-Net-STM  –  Parameter Summary                     ║
╠══════════════════╦═══════════════════════╦══════════════════╣
║  Layer           ║  Type                 ║           Params  ║
╠══════════════════╬═══════════════════════╬══════════════════╣
║  conv2d          ║  Conv2D               ║              288  ║
║  conv2d_1        ║  Conv2D               ║              160  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  batch_normaliz  ║  BatchNormalization   ║              128  ║
║  dense           ║  Dense              

I0000 00:00:1777120122.008190     152 service.cc:152] XLA service 0x784e40527f80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777120122.008243     152 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1777120122.008249     152 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1777120125.629197     152 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1777120147.427965     152 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1100/1100 ━━━━━━━━━━━━━━━━━━━━ 183s 124ms/step - accuracy: 0.7223 - loss: 1.6386 - val_accuracy: 0.9698 - val_loss: 0.8218
Epoch 2/100
1100/1100 ━━━━━━━━━━━━━━━━━━━━ 101s 89ms/step - accuracy: 0.9842 - loss: 0.7751 - val_accuracy: 0.9637 - val_loss: 0.8274
Epoch 3/100
1100/1100 ━━━━━━━━━━━━━━━━━━━━ 107s 95ms/step - accuracy: 0.9885 - loss: 0.7489 - val_accuracy: 0.9838 - val_loss: 0.7604
Epoch 4/100
1100/1100 ━━━━━━━━━━━━━━━━━━━━ 105s 92ms/step - accuracy: 0.9912 - loss: 0.7356 - val_accuracy: 0.9884 - val_loss: 0.7409
Epoch 5/100
1100/1100 ━━━━━━━━━━━━━━━━━━━━ 108s 95ms/step - accuracy: 0.9925 - loss: 0.7278 - val_accuracy: 0.9909 - val_loss: 0.7336
Epoch 6/100
1100/1100 ━━━━━━━━━━━━━━━━━━━━ 104s 91ms/step - accuracy: 0.9939 - loss: 0.7213 - val_accuracy: 0.9903 - val_loss: 0.7352
Epoch 7/100
1100/1100 ━━━━━━━━━━━━━━━━━━━━ 106s 93ms/step - accuracy: 0.9946 - loss: 0.7186 - val_accuracy: 0.9950 - val_loss: 0.7163
Epoch 8/100
1100/1100 ━━━━━━━━━━━━━━━━━━━━ 104s 92ms/step - accuracy: 0.9